In [0]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalog", "insurance_project")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

for value in [catalog, silver_schema, gold_schema]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid catalog/schema name: {value}")

s = f"{catalog}.{silver_schema}"   # silver prefix used in your SQL
g = f"{catalog}.{gold_schema}"     # gold prefix

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {g}")

def save_gold(table_name, query):
    df = spark.sql(query)
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{g}.{table_name}")
    )
    print(f"Saved {g}.{table_name}: {df.count()} rows")

In [0]:
save_gold(
    "dim_customer",
    f"""
    SELECT
        customer_id,
        concat_ws(' ', first_name, last_name) AS full_name,
        date_of_birth,
        gender,
        marital_status,
        occupation,
        annual_income,
        city,
        state,
        country,
        registration_date
    FROM {s}.customers
    """
)

save_gold(
    "dim_product",
    f"""
    SELECT
        product_id,
        product_name,
        insurance_type,
        plan_tier,
        coverage_limit,
        annual_premium,
        deductible_amount,
        currency,
        policy_term_days,
        is_active
    FROM {s}.insurance_products
    """
)

save_gold(
    "dim_agent",
    f"""
    SELECT
        a.agent_id,
        a.agent_name,
        a.agent_type,
        a.commission_rate_percent,
        a.is_active,
        a.branch_id,
        b.branch_name,
        b.city AS branch_city,
        b.state AS branch_state
    FROM {s}.agents a
    LEFT JOIN {s}.branches b
      ON a.branch_id = b.branch_id
    """
)


In [0]:
save_gold(
    "fact_claim",
    f"""
    WITH settlement_totals AS (
        SELECT
            claim_id,
            COUNT(*) AS settlement_installment_count,
            SUM(paid_amount) AS paid_amount,
            MAX(settlement_date) AS last_settlement_date
        FROM {s}.claim_settlements
        GROUP BY claim_id
    )
    SELECT
        c.claim_id,
        c.policy_id,
        c.customer_id,
        p.product_id,
        p.agent_id,
        pr.insurance_type,
        c.incident_date,
        c.claim_date,
        c.claim_status,
        c.claimed_amount,
        a.decision AS assessment_decision,
        a.approved_amount,
        COALESCE(
            st.paid_amount,
            CAST(0 AS DECIMAL(18,2))
        ) AS paid_amount,
        COALESCE(
            st.settlement_installment_count, 0
        ) AS settlement_installment_count,
        st.last_settlement_date,
        CASE
            WHEN a.approved_amount IS NULL THEN NULL
            ELSE a.approved_amount
                 - COALESCE(st.paid_amount, 0)
        END AS approved_unpaid_amount,
        c.currency
    FROM {s}.claims c
    INNER JOIN {s}.policies p
      ON c.policy_id = p.policy_id
    INNER JOIN {s}.insurance_products pr
      ON p.product_id = pr.product_id
    LEFT JOIN {s}.claim_assessments a
      ON c.claim_id = a.claim_id
    LEFT JOIN settlement_totals st
      ON c.claim_id = st.claim_id
    """
)


In [0]:
save_gold(
    "fact_policy",
    f"""
    WITH premium_totals AS (
        SELECT
            policy_id,
            SUM(premium_due) AS premium_due,
            SUM(amount_paid) AS premium_collected,
            SUM(outstanding_amount) AS premium_outstanding
        FROM {s}.premium_payments
        GROUP BY policy_id
    ),
    claim_totals AS (
        SELECT
            policy_id,
            COUNT(*) AS claim_count,
            SUM(claimed_amount) AS claimed_amount,
            SUM(paid_amount) AS claims_paid
        FROM {g}.fact_claim
        GROUP BY policy_id
    )
    SELECT
        p.policy_id,
        p.customer_id,
        p.product_id,
        p.agent_id,
        pr.insurance_type,
        c.city AS customer_city,
        c.state AS customer_state,
        p.start_date,
        p.end_date,
        p.policy_status,
        p.sum_insured,
        p.annual_premium,
        COALESCE(pt.premium_due, 0) AS premium_due,
        COALESCE(pt.premium_collected, 0) AS premium_collected,
        COALESCE(pt.premium_outstanding, 0) AS premium_outstanding,
        COALESCE(ct.claim_count, 0) AS claim_count,
        COALESCE(ct.claimed_amount, 0) AS claimed_amount,
        COALESCE(ct.claims_paid, 0) AS claims_paid,
        p.currency
    FROM {s}.policies p
    INNER JOIN {s}.insurance_products pr
      ON p.product_id = pr.product_id
    INNER JOIN {s}.customers c
      ON p.customer_id = c.customer_id
    LEFT JOIN premium_totals pt
      ON p.policy_id = pt.policy_id
    LEFT JOIN claim_totals ct
      ON p.policy_id = ct.policy_id
    """
)


In [0]:
save_gold(
    "insurance_portfolio_summary",
    f"""
    SELECT
        insurance_type,
        COUNT(*) AS policy_count,
        COUNT(DISTINCT customer_id) AS customer_count,
        SUM(annual_premium) AS annual_premium,
        SUM(premium_collected) AS premium_collected,
        SUM(premium_outstanding) AS premium_outstanding,
        SUM(claim_count) AS claim_count,
        SUM(claimed_amount) AS claimed_amount,
        SUM(claims_paid) AS claims_paid,
        CASE
            WHEN SUM(premium_collected) = 0 THEN NULL
            ELSE ROUND(
                100.0 * SUM(claims_paid)
                / SUM(premium_collected),
                2
            )
        END AS paid_claims_to_collected_premium_pct
    FROM {g}.fact_policy
    GROUP BY insurance_type
    """
)

save_gold(
    "claims_monthly_summary",
    f"""
    SELECT
        CAST(date_trunc('MONTH', claim_date) AS DATE)
            AS claim_month,
        insurance_type,
        COUNT(*) AS claim_count,
        SUM(
            CASE WHEN claim_status = 'Settled'
                 THEN 1 ELSE 0 END
        ) AS settled_claim_count,
        SUM(
            CASE WHEN claim_status = 'Rejected'
                 THEN 1 ELSE 0 END
        ) AS rejected_claim_count,
        SUM(
            CASE WHEN claim_status = 'Under Review'
                 THEN 1 ELSE 0 END
        ) AS under_review_claim_count,
        SUM(claimed_amount) AS claimed_amount,
        SUM(approved_amount) AS known_approved_amount,
        SUM(paid_amount) AS paid_amount
    FROM {g}.fact_claim
    GROUP BY
        CAST(date_trunc('MONTH', claim_date) AS DATE),
        insurance_type
    """
)


In [0]:
from decimal import Decimal

policy_fact = spark.table(f"{g}.fact_policy")
claim_fact = spark.table(f"{g}.fact_claim")

assert policy_fact.count() == 100000
assert claim_fact.count() == 100000

assert (
    policy_fact.select("policy_id").distinct().count()
    == 100000
)

assert (
    claim_fact.select("claim_id").distinct().count()
    == 100000
)

premium_total = policy_fact.agg(
    F.sum("premium_collected").alias("value")
).first()["value"]

claimed_total = claim_fact.agg(
    F.sum("claimed_amount").alias("value")
).first()["value"]

paid_total = claim_fact.agg(
    F.sum("paid_amount").alias("value")
).first()["value"]

assert premium_total == Decimal("1410000000.00")
assert claimed_total == Decimal("10950000000.00")
assert paid_total == Decimal("2650000000.00")

print("Gold validation passed")
print("Premium collected:", premium_total)
print("Claimed amount:", claimed_total)
print("Claims paid:", paid_total)

display(
    spark.table(f"{g}.insurance_portfolio_summary")
    .orderBy("insurance_type")
)

display(
    claim_fact.groupBy("claim_status")
    .count()
    .orderBy("claim_status")
)


In [0]:
%sql
DESCRIBE DETAIL insurance_project.bronze.customers;


In [0]:
%sql
DESCRIBE DETAIL insurance_project.silver.customers;



In [0]:
%sql
DESCRIBE DETAIL insurance_project.gold.fact_claim;